## Data provenance and integrity requirements

This notebook refuses missing scan roots by default. Test fixtures are allowed only when `WWGPT_ALLOW_TEST_FIXTURE=1`. Immediate alpha analysis requires `immediate_spectral_source == 'weightwatcher_measured'` and `measurement_valid_for_science == true`; legacy fabricated or unverified files are excluded. Run `wwgpt audit-strength-scan --scan-root PATH` before analysis.


In [ ]:
import os
from pathlib import Path
SCAN_ROOT = Path(os.environ.get('WWGPT_STRENGTH_SCAN_ROOT', 'results/latest_strength_scan'))
if not SCAN_ROOT.exists():
    if os.environ.get('WWGPT_ALLOW_TEST_FIXTURE') == '1':
        SCAN_ROOT = Path('tests/fixtures/strength_scan')
    else:
        raise FileNotFoundError(f'Requested strength scan root does not exist: {SCAN_ROOT}')
print('scan root:', SCAN_ROOT)
print('fixture allowed:', os.environ.get('WWGPT_ALLOW_TEST_FIXTURE') == '1')


In [ ]:
import os
import pandas as pd
spectral_files = list(SCAN_ROOT.rglob('wwpgd_projection_spectral.csv'))
for path in spectral_files:
    df = pd.read_csv(path)
    ok = {'immediate_spectral_source','measurement_valid_for_science'}.issubset(df.columns) and (df['immediate_spectral_source'] == 'weightwatcher_measured').all() and df['measurement_valid_for_science'].astype(str).str.lower().isin(['true','1']).all()
    if not ok:
        if os.environ.get('WWGPT_ALLOW_TEST_FIXTURE') == '1':
            print(f'WARNING: excluding legacy_fabricated_or_unverified fixture file: {path}')
        else:
            raise ValueError(f'Immediate alpha analysis refused legacy_fabricated_or_unverified file: {path}')


# Strength Scan WeightWatcher

## Configuration

In [ ]:
from pathlib import Path
import os

SCAN_ROOT = Path(
    os.environ.get(
        "WWGPT_STRENGTH_SCAN_ROOT",
        "/tmp/wwpgd_strength_scan",
    )
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wwgpt.strength_scan_analysis import resolve_scan_root, analyze_strength_scan
try:
    scan_root = resolve_scan_root(SCAN_ROOT)
except FileNotFoundError:
    scan_root = resolve_scan_root(Path("tests/fixtures/strength_scan"))
analysis_dir = analyze_strength_scan(scan_root)
fig_dir = scan_root / "analysis" / "notebook_weightwatcher"
fig_dir.mkdir(parents=True, exist_ok=True)
print(scan_root)

## Immediate spectral-record audit

## Fit-quality audit

## Alpha before and after projection

## Distance to target alpha

## Per-layer response

## Projection-event response

## Projection magnitude versus spectral correction

## TraceLog preservation

## Strength-response interpretation

In [ ]:
try:
    spec=pd.read_csv(analysis_dir/"strength_scan_spectral.csv")
except pd.errors.EmptyDataError:
    spec=pd.DataFrame()
runs=pd.read_csv(analysis_dir/"strength_scan_runs.csv")
raw=[]
for d in runs[runs.optimizer_family=="wwpgd"].run_dir:
    p=Path(d)/"wwpgd_projection_spectral.csv"
    if p.exists(): raw.append(pd.read_csv(p))
raw=pd.concat(raw, ignore_index=True) if raw else pd.DataFrame()
display(raw); display(raw[raw[["alpha_before","alpha_after"]].isna().any(axis=1)] if len(raw) else raw); display(spec)
if len(raw):
    ax=raw.plot.scatter(x="alpha_before", y="alpha_after", c="scan_strength", colormap="viridis"); ax.figure.savefig(fig_dir/"alpha_before_after.png"); plt.close(ax.figure)
    for y,name in [("abs_alpha_error_change","alpha_error_change"),("relative_frobenius_change","fro_vs_alpha")]:
        ax=raw.plot.scatter(x="scan_strength" if y=="abs_alpha_error_change" else "relative_frobenius_change", y=y); ax.figure.savefig(fig_dir/f"{name}.png"); plt.close(ax.figure)
    for y,name in [("TraceLog_change","tracelog_change"),("D_before","fit_D_before"),("num_evals_before","eligible_fits"),("effective_blend_eta","etas")]:
        ax=raw.groupby("scan_strength")[y].median().plot(marker="o"); ax.figure.savefig(fig_dir/f"{name}.png"); plt.close(ax.figure)
if len(spec):
    for y,name in [("median_abs_alpha_error_change","median_error_change"),("fraction_layers_closer_to_target","fraction_closer"),("median_alpha_before","median_alpha_event")]:
        ax=spec.groupby("strength")[y].mean().plot(marker="o"); ax.figure.savefig(fig_dir/f"{name}.png"); plt.close(ax.figure)
print("Fit filtering is explicit; invalid fits remain in audit tables. Layers are not treated as independent seeds.")